# Preprocessing

Paper: Appendix A.1 — removes problems that rely on the answer choices to be solvable (e.g. "which of the following is correct?"), which would otherwise give distractor generation an unbounded answer space (Section 4: 429 Eedi / 500 SciQ problems after filtering).

We filter such problems, using an LLM to flag likely-unsolvable candidates (many are trivially unsolvable) and manually reviewing the rest.

In [ ]:
import os
from typing import List, Dict, Tuple
import json
import re
from tqdm import tqdm

from dotenv import load_dotenv
from openai import OpenAI

import scipy.stats as st
import pandas as pd
import numpy as np

from src.datasets import get_or_create_dataset

from src.prompt_util import prompt_openai
from src.model_configurations import gpt_5_mini_config

load_dotenv()

## EEDI

In [ ]:
data_folder = "eedi_data"
dataset = get_or_create_dataset(data_folder, n_limit=5000)
print(f"We have {len(dataset)} questions")

### Prompting

In [ ]:
model_config = gpt_5_mini_config
client = OpenAI(base_url=model_config["base_url"], api_key=os.environ.get(model_config["api_key_var"], None))

output_folder = os.path.join(data_folder, f"{model_config['model']}_temp_{model_config['completion_kwargs']['temperature']}_distr_annot")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f)

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

In [ ]:
# Ask the LLM to judge whether a problem is solvable
fewshot_solvable = (
    "Question: What is 5 + 7?\n"
    "Is this math problem solvable by a student with the information given? Answer yes or no.\n"
    "A: yes\n\n"
    "Question: What is the mode of the following numbers? \[ 1,1,4,6,7,7 \]\n"
    "Is this math problem solvable by a student with the information given? Answer yes or no.\n"
    "A: yes\n\n"
    "Question: Which of the numbers is the same when rounded to \( 2 \) decimal places or to \( 2 \) significant figures?\n"
    "Is this math problem solvable by a student with the information given? Answer yes or no.\n"
    "A: no\n\n"
)

problem_solvable_by_datapointid = {}
for i in tqdm(range(len(dataset))):
    problem = dataset[i]["Problem"]
    prompt = (
        fewshot_solvable +
        f"Question: {problem['Question']}\n"
        "Is this math problem solvable by a student with the information given? Answer yes or no."
    )
    response = prompt_openai(client, "You are a math teacher.", prompt, model_config)
    problem_solvable_by_datapointid[str(i)] = ("yes" in response.lower())
    if i % 10 == 0:
        save_json(problem_solvable_by_datapointid, os.path.join(output_folder, "problem_solvable_by_datapointid.json"))
save_json(problem_solvable_by_datapointid, os.path.join(output_folder, "problem_solvable_by_datapointid.json"))

In [ ]:
problem_solvable_by_datapointid = load_json(os.path.join(output_folder, "problem_solvable_by_datapointid.json"))

### Full Manual Inspection

In [ ]:
# inspect problem solvable
for dpid,dp in enumerate(dataset):
    print(dpid)
    print(dp["Problem"]["Question"])
    print(problem_solvable_by_datapointid[str(dpid)])
    print("-"*30)

In [ ]:
# correction after manual inspection
for dpid in [20, 43, 49, 100, 129, 162, 247, 511]:
    problem_solvable_by_datapointid[str(dpid)] = False

for dpid in [54, 175, 183, 185, 190, 223, 254, 300, 311, 312, 349, 362, 364, 401, 408, 463, 468, 500]:
    problem_solvable_by_datapointid[str(dpid)] = True

save_json(problem_solvable_by_datapointid, os.path.join(output_folder, "problem_solvable_by_datapointid_corrected.json"))

In [ ]:
from collections import Counter
Counter(problem_solvable_by_datapointid.values())

### Convert From DatapointId to QuestionId

In [ ]:
# Final Eedi solvability annotations, keyed by question id (Appendix A.1)
import json

with open("eedi_data/gpt-5-mini-2025-08-07_temp_1.0_distr_annot/problem_solvable_by_datapointid_corrected.json", "r") as f:
    data = json.load(f)

data_by_questionid = {dataset.questionids[int(k)]: v for k,v in data.items()}
with open("eedi_data/problem_solvable_by_questionid_corrected.json", "w") as f:
    json.dump(data_by_questionid, f)

## SciQ

Filter SciQ questions where the question itself is unsolvable without the answer choices (e.g. "which of the following…"). Hard-filter obvious patterns via regex, then manually review the rest until 500 solvable questions are confirmed.

NOTE: no hard patterns observed

In [ ]:
import random
from src.datasets import SciQDataset

sciq_data_folder = "sciq_data"

# Load the full train split from HuggingFace (no n_limit — we need to review all candidates)
sciq_full = SciQDataset(data_folder=sciq_data_folder, split="train")
print(f"Total SciQ train questions: {len(sciq_full)}")

### Auto-filter

In [ ]:
UNSOLVABLE_PATTERNS = [
    r"\bwhich of the following\b",
    r"\bwhich one of the following\b",
    r"\ball of the following\b",
    r"\bnone of the following\b",
    r"\bnone of the above\b",
    r"\ball of the above\b",
    r"\bwhich .{0,40}(is|are) (not |in)?correct\b",
    r"\bwhich .{0,40}(is|are) (not )?true\b",
    r"\bwhich .{0,40}(is|are) (not )?false\b",
    r"\bwhich .{0,40}best describes\b",
    r"\bwhich .{0,40}best explains\b",
]

solvable_by_qid = {}
for qid in sciq_full.questionids:
    question = sciq_full.items_by_qid[qid]["question"]
    auto_unsolvable = any(re.search(p, question, re.IGNORECASE) for p in UNSOLVABLE_PATTERNS)
    solvable_by_qid[qid] = not auto_unsolvable

n_solvable = sum(solvable_by_qid.values())
n_total = len(solvable_by_qid)
print(f"Auto-filtered as unsolvable: {n_total - n_solvable}/{n_total}")
print(f"Remaining candidates:        {n_solvable}/{n_total}")

### Manual Inspection

Print candidates in random order. Mark questions as unsolvable in the correction cell below if they require the answer choices to be answerable.

In [ ]:
confirmed_solvable_qids = []
confirmed_unsolable_qids = []

In [ ]:
random.seed(42)
candidates = [qid for qid, s in solvable_by_qid.items() if s and qid not in confirmed_solvable_qids and qid not in confirmed_unsolable_qids]
random.shuffle(candidates)

BATCH = candidates[:20]

for qid in BATCH:
    print(qid)
    print(sciq_full.items_by_qid[qid]["question"])
    print("-" * 30)

In [ ]:
# Corrections after manual inspection
unsolvable_of_batch = ["sciq_train_6623", "sciq_train_10388"]
confirmed_solvable_qids.extend([qid for qid in BATCH if qid not in unsolvable_of_batch])
confirmed_unsolable_qids.extend(unsolvable_of_batch)

In [ ]:
len(confirmed_solvable_qids)

In [ ]:
solvable_by_qid = {qid: True for qid in confirmed_solvable_qids}

for qid in confirmed_unsolable_qids:
    solvable_by_qid[qid] = False

n_solvable = sum(solvable_by_qid.values())
print(f"Confirmed solvable after corrections: {n_solvable}")

### Save Annotations and Create dataset-500.json

In [ ]:
# Persist solvability annotations, then sample the 500 SciQ questions used throughout the paper (Section 4)
solvable_path = os.path.join(sciq_data_folder, "problem_solvable_by_questionid.json")
with open(solvable_path, "w") as f:
    json.dump(solvable_by_qid, f)
print(f"Saved solvability annotations to {solvable_path}")

# Select 500 solvable questions at random and save dataset-500.json
solvable_qids = [qid for qid, s in solvable_by_qid.items() if s]
assert len(solvable_qids) >= 500, f"Only {len(solvable_qids)} solvable questions - review more candidates first"

random.seed(42)
selected_qids = random.sample(solvable_qids, 500)

sciq_dataset_500 = SciQDataset(
    data_folder=sciq_data_folder,
    split="train",
    n_limit=500,
    items_by_qid=sciq_full.items_by_qid,
    questionids=selected_qids,
)
dataset_path = os.path.join(sciq_data_folder, "dataset-500.json")
sciq_dataset_500.save(dataset_path)
print(f"Saved {len(sciq_dataset_500)} questions to {dataset_path}")